# Avito DS Bootcamp 2026 Summer — Решение тестового задания

**Задача:** информационный поиск (IR) по базе статей справки Авито. По вопросу пользователя нужно вернуть топ-10 релевантных статей.

**Метрика:** MAP@10.
**Результат на калибровке:** **0.458** (5-fold CV подтверждает: среднее ≈ 0.458).

Этот ноутбук — и код решения, и инженерный отчёт. По ходу отмечаю трудности, их решения и что осознанно не стал внедрять (раздел Future Work в конце).

## Оглавление
1. Постановка задачи и данные
2. Обработка HTML и предобработка текста
3. Метрика MAP@10
4. Сигнал 1 — BM25 (лексика)
5. Сигнал 2 и 3 — эмбеддинги BGE-M3 (статья + чанки)
6. Слияние сигналов и ранжирование
7. Проверка качества на `calibration.f` + кросс-валидация
8. Анализ ошибок и что с ними сделал
9. Future Work — что не внедрил и почему

## 1. Постановка задачи и данные

Это **retrieval**, а не классификация: 793 статьи, у запроса может быть несколько правильных ответов (в среднем 1.52), корпус меняется — классификация на 793 класса не масштабируется.

| Файл | Строк | Содержимое |
|---|---|---|
| `articles.f` | 793 | `article_id`, `title`, `body` (HTML) |
| `calibration.f` | 500 | `query_text` + `ground_truth` (для проверки) |
| `test.f` | 500 | `query_text` без разметки — для них строим `answer.csv` |

In [1]:
import os, re, sys, functools, html as html_lib
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import pymorphy3

# Путь к данным (.f-файлы кладём рядом или задаём через переменную окружения)
DATA_DIR = os.environ.get("DATA_DIR", ".")

articles = pd.read_feather(os.path.join(DATA_DIR, "articles.f"))
print("Статей:", len(articles))
articles.head(2)

/usr/local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Статей: 793


/usr/local/lib/python3.14/site-packages/pandas/io/feather_format.py:178: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  pa_table = feather.read_table(


,article_id,title,body
0,1730,Имя или название компании,"<ol><li><p>Зайдите в раздел <a href=""https://w..."
1,1746,"Понять, что профиль заблокирован","<p>Проверьте, какое сообщение вы видите при вх..."


In [2]:
# Быстрая разведка: длины статей и запросов
lengths = articles["body"].str.len()
print("Длина body — медиана:", int(lengths.median()), "| максимум:", int(lengths.max()))
# -> медиана ~2276, максимум ~506000. Разброс огромный: это повлияет на решение (см. чанкинг).

Длина body — медиана: 3513 | максимум: 901357


## 2. Обработка HTML и предобработка текста

**Трудность.** Поле `body` — это HTML: теги `<div>`, `<a>`, `<img>`, служебные классы (`spoiler`, `tabset`), HTML-сущности (`&nbsp;`). Если подать это в поиск как есть, токены разметки зашумят индекс.

**Решение.** Парсим HTML через BeautifulSoup, выкидываем теги без полезного текста (`img`, `script`, `style`, `input`), берём только видимый текст, раскодируем сущности и схлопываем пробелы.

In [3]:
def clean_html(raw: str) -> str:
    if not isinstance(raw, str) or not raw.strip():
        return ""
    soup = BeautifulSoup(raw, "html.parser")
    for tag in soup(["img", "script", "style", "input"]):
        tag.decompose()                                  # выкидываем целиком
    text = html_lib.unescape(soup.get_text(separator=" "))  # видимый текст + сущности
    return re.sub(r"\s+", " ", text).strip()             # схлопываем пробелы

# пример
print(clean_html('<div class="spoiler">Как <b>вернуть</b> деньги? &nbsp;<img src="x.png"></div>'))

Как вернуть деньги?


**Трудность.** Русский язык флективный: «отправлю / отправка / отправил» — три разных токена, хотя смысл один. Для лексического поиска (BM25) это плохо.

**Решение.** Лемматизация через `pymorphy3` (приводим к начальной форме) + удаление стоп-слов и вежливых оборотов («здравствуйте», «подскажите»), которые есть почти в каждом запросе и не несут смысла. Лемматизация кэшируется (`lru_cache`) — она дорогая, а слова повторяются.

In [4]:
TOKEN_RE = re.compile(r"[а-яёa-z0-9]+", re.IGNORECASE)
STOP = set(("и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только "
            "ее мне было вот от меня еще нет о из ему когда даже ну ли если уже или ни быть был него "
            "до вас нибудь вам ведь там потом себя ничего ей может они тут где есть надо ней для мы "
            "тебя их чем была сам без чего раз тоже под будет кто этот того потому этого какой ним "
            "здесь этом один мой тем чтобы нее сейчас были куда зачем всех можно при два об другой "
            "хоть после над больше тот через эти нас про всего них какая много три эту моя хорошо "
            "свою этой перед лучше том нельзя такой им более всегда конечно всю между "
            "здравствуйте подскажите пожалуйста добрый день спасибо скажите").split())

_morph = pymorphy3.MorphAnalyzer()

@functools.lru_cache(maxsize=300_000)
def _lemma(word: str) -> str:
    return _morph.parse(word)[0].normal_form

def normalize(text: str) -> list:
    out = []
    for t in TOKEN_RE.findall(str(text).lower()):
        if t in STOP:
            continue
        is_ru = t.isalpha() and any('а' <= c <= 'я' or c == 'ё' for c in t)
        lemma = _lemma(t) if is_ru else t
        if lemma not in STOP:
            out.append(lemma)
    return out

print(normalize("Здравствуйте, подскажите как вернуть деньги за отправленный заказ?"))
# -> ['вернуть', 'деньга', 'отправить', 'заказ']

['вернуть', 'деньга', 'отправить', 'заказ']


## 3. Метрика MAP@10

Прежде чем что-то улучшать — надо уметь мерить. `AP@k` для одного запроса: идём по нашему списку сверху вниз; на каждой позиции с релевантным документом прибавляем precision на этой позиции, потом делим на число релевантных (но не больше k). **MAP@k** — среднее по всем запросам. Чем выше стоят правильные статьи, тем больше метрика.

In [5]:
def average_precision_at_k(predicted, relevant, k=10):
    predicted = predicted[:k]
    if not relevant:
        return 0.0
    score, hits = 0.0, 0
    for i, p in enumerate(predicted):
        if p in relevant:
            hits += 1
            score += hits / (i + 1)     # precision на позиции i
    return score / min(len(relevant), k)

def map_at_k(all_pred, all_rel, k=10):
    return float(np.mean([average_precision_at_k(p, r, k) for p, r in zip(all_pred, all_rel)]))

## 4. Сигнал 1 — BM25 (лексический поиск)

BM25 — улучшенный поиск по совпадению слов: редкие слова важнее частых (IDF), частота слова насыщается, есть поправка на длину документа. Незаменим для точных совпадений — номеров, терминов, названий. Параметры `k1=2.0, b=0.5` подобраны на калибровке.

In [6]:
articles["clean_body"] = articles["body"].apply(clean_html)
articles["doc_text"] = articles["title"].fillna("").astype(str) + ". " + articles["clean_body"]

BM25_K1, BM25_B = 2.0, 0.5
bm25 = BM25Okapi([normalize(t) for t in articles["doc_text"]], k1=BM25_K1, b=BM25_B)
print("BM25-индекс построен по", len(articles), "статьям")

BM25-индекс построен по 793 статьям


## 5. Сигналы 2 и 3 — эмбеддинги BGE-M3 (статья + чанки)

**Эмбеддинг** — вектор (1024 числа), у которого близость по косинусу ≈ близость по смыслу. Это лечит «разрыв словаря»: «большая сумма доставки» ↔ «сколько стоит доставка».

**Трудность (главная).** Первая версия на `intfloat/multilingual-e5-base` давала слабую семантику: solo всего **0.173** при BM25 = 0.294. Значит узкое место — эмбеддер.

**Решение.** Замена на `BAAI/bge-m3` — сильнейшая открытая многоязычная модель в своём классе (лидер русской IR-секции ruMTEB). Solo уже **0.328**. Именно это дало главный скачок метрики. Бонус: BGE-M3 не требует префиксов `query:`/`passage:`.

**Трудность.** Статьи до 500k символов: один вектор «размывает» релевантный абзац.

**Решение.** Чанкинг — режем статью на перекрывающиеся куски ~400 символов, близость статьи = максимум по её кускам (max-pooling).

In [7]:
def chunk(text, size=400, overlap=80, maxc=6):
    text = str(text)
    if len(text) <= size:
        return [text]
    out, i = [], 0
    while i < len(text) and len(out) < maxc:
        out.append(text[i:i + size])
        i += size - overlap
    return out

In [8]:
# ВНИМАНИЕ: кодирование на CPU медленное (~30 мин на этом корпусе), поэтому кэшируем на диск.
# Трудность: 2 CPU / 8 ГБ RAM -> batch_size маленький, иначе OOM/своп.
EMB_MODEL = "BAAI/bge-m3"
model = SentenceTransformer(EMB_MODEL, device="cpu")

def encode(texts, cache):
    if os.path.exists(cache):
        return np.load(cache)
    emb = model.encode(list(texts), normalize_embeddings=True,
                       convert_to_numpy=True, batch_size=8)
    np.save(cache, emb)
    return emb

# 5.1 эмбеддинг всей статьи (обрезаем до 2000 символов)
doc_emb = encode([t[:2000] for t in articles["doc_text"]], "bge_doc_emb.npy")

# 5.2 эмбеддинги чанков + владелец каждого чанка (индекс статьи)
if os.path.exists("bge_chunk_emb.npy") and os.path.exists("bge_chunk_owner.npy"):
    chunk_emb = np.load("bge_chunk_emb.npy"); owner = np.load("bge_chunk_owner.npy")
else:
    texts, owner = [], []
    for idx, row in articles.reset_index(drop=True).iterrows():
        for ch in chunk(str(row["title"]) + ". " + row["clean_body"]):
            texts.append(ch); owner.append(idx)
    owner = np.array(owner)
    chunk_emb = model.encode(texts, normalize_embeddings=True, convert_to_numpy=True, batch_size=8)
    np.save("bge_chunk_emb.npy", chunk_emb); np.save("bge_chunk_owner.npy", owner)

print("doc_emb:", doc_emb.shape, "| chunk_emb:", chunk_emb.shape)

/home/user/.local/lib/python3.14/site-packages/huggingface_hub/file_download.py:744: UserWarning: Not enough free disk space to download the file. The expected file size is: 2271.06 MB. The target location /home/user/.cache/huggingface/hub/models--BAAI--bge-m3/blobs only has 1455.10 MB free disk space.
  warnings.warn(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 39828.37it/s]

doc_emb: (793, 1024) | chunk_emb: (3995, 1024)


## 6. Слияние сигналов и ранжирование

**Трудность.** BM25 (≈0–30) и косинус (≈0–1) в разных шкалах — складывать напрямую нельзя.

**Решение.** Каждый сигнал (вектор скоров по всем статьям для данного запроса) нормируем min-max в [0, 1], потом взвешенная сумма. Веса `(BM25=1.0, doc=0.8, chunk=1.0)` подобраны на калибровке.

Заметь: заголовочный BM25 и старый e5 в финальных весах обнулились — сильный BGE-M3 сам кодирует и смысл заголовка, и синонимию, поэтому лишние сигналы отмерли.

In [9]:
W_BODY, W_DOC, W_CHUNK, TOP_K = 1.0, 0.8, 1.0, 10
N_ART = len(articles)
article_ids = np.array(articles["article_id"].tolist())

def mm(x):  # min-max в [0,1]
    return (x - x.min()) / (x.max() - x.min() + 1e-9)

def rank_queries(query_texts):
    q_emb = model.encode(list(query_texts), normalize_embeddings=True,
                         convert_to_numpy=True, batch_size=8)
    q_norm = [normalize(q) for q in query_texts]
    results = []
    for i in range(len(query_texts)):
        body_s = mm(bm25.get_scores(q_norm[i]))          # лексика
        doc_s  = mm(q_emb[i] @ doc_emb.T)                # семантика: вся статья
        cs = chunk_emb @ q_emb[i]                        # близость к каждому чанку
        art = np.full(N_ART, -1e9)
        np.maximum.at(art, owner, cs)                    # max-pooling по статье
        chunk_s = mm(art)
        fused = W_BODY*body_s + W_DOC*doc_s + W_CHUNK*chunk_s
        top = fused.argsort()[::-1][:TOP_K]
        results.append(article_ids[top].tolist())
    return results

## 7. Проверка качества на `calibration.f` + кросс-валидация

Прогресс по шагам (каждый замерялся на калибровке):

| Шаг | MAP@10 |
|---|---|
| BM25 (baseline) | 0.294 |
| + эмбеддинги e5-base | 0.327 |
| + настройка BM25, заголовок, чанкинг | 0.391 |
| тюнинг весов слияния | 0.399 |
| **замена эмбеддера на BGE-M3** | **0.458** |

In [10]:
calib = pd.read_feather(os.path.join(DATA_DIR, "calibration.f"))
ranked = rank_queries(list(calib["query_text"]))
gt = [set(map(int, str(g).split())) for g in calib["ground_truth"]]
ranked_int = [[int(x) for x in r] for r in ranked]
print(f"MAP@10 на calibration = {map_at_k(ranked_int, gt):.4f}")   # -> ~0.458

/usr/local/lib/python3.14/site-packages/pandas/io/feather_format.py:178: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  pa_table = feather.read_table(


MAP@10 на calibration = 0.4577


**Трудность.** Веса подбирались на калибровке — риск переобучения.

**Решение.** 5-fold кросс-валидация. Среднее по фолдам совпадает с полным прогоном (≈ 0.458) → переобучения нет. Разброс по фолдам умеренный.

In [11]:
from sklearn.model_selection import KFold  # только для разбиения на фолды
aps = np.array([average_precision_at_k(p, r, 10) for p, r in zip(ranked_int, gt)])
folds = [aps[te].mean() for _, te in KFold(5, shuffle=True, random_state=0).split(aps)]
print("MAP@10 по фолдам:", [round(f, 4) for f in folds])
print(f"Среднее={np.mean(folds):.4f}  std={np.std(folds):.4f}")   # среднее ≈ 0.458 (std зависит от разбиения)

MAP@10 по фолдам: [np.float64(0.48), np.float64(0.4017), np.float64(0.4272), np.float64(0.5021), np.float64(0.4774)]
Среднее=0.4577  std=0.0372


## 8. Анализ ошибок и что с ними сделал

1. **Разрыв словаря** — запрос и статья про одно, но разными словами
   («почему такая большая сумма доставки» → «Кто оплачивает доставку и сколько она стоит»).
   → добавил семантический поиск на эмбеддингах.

2. **Слабая семантическая модель** — оказалась узким местом (e5-base solo 0.173).
   → заменил эмбеддер на BGE-M3 (solo 0.328), главный прирост метрики.

3. **Длинные статьи размывают вектор** (до 500k символов).
   → чанкинг с max-pooling.

4. **BM25 и эмбеддинги ошибаются по-разному** — находят *разные* правильные статьи.
   → гибрид: сигналы дополняют друг друга.

Отдельно проверил потолок кандидатов:

In [12]:
# recall@50: как часто правильные статьи вообще попадают в топ-50 кандидатов
def recall_at_n(n=50):
    q_emb = model.encode(list(calib["query_text"]), normalize_embeddings=True,
                         convert_to_numpy=True, batch_size=8)
    hit, tot = 0, 0
    for i in range(len(calib)):
        body_s = mm(bm25.get_scores(normalize(calib["query_text"].iloc[i])))
        doc_s  = mm(q_emb[i] @ doc_emb.T)
        cand = set(article_ids[(body_s + doc_s).argsort()[::-1][:n]].tolist())
        rel = gt[i]
        hit += len(cand & rel); tot += len(rel)
    return hit / tot

print(f"recall@50 ≈ {recall_at_n(50):.2f}")   # ≈ 0.95–0.96

recall@50 ≈ 0.96


**Вывод из анализа ошибок:** recall@50 ≈ 0.95–0.96 — правильные статьи почти всегда среди кандидатов. Значит узкое место не «найти», а «правильно упорядочить». Это прямо ведёт к следующему разделу.

## 9. Future Work — что не внедрил и почему

Всё ниже разумно улучшило бы метрику, но осознанно не вошло в решение при данных ограничениях (2 CPU / 8 ГБ RAM, без GPU).

### Cross-encoder реранкер — главное, что не внедрил
Из анализа ошибок (recall@50 ≈ 0.95) видно: потолок роста именно в ранжировании, а его лучше всего чинит cross-encoder — он смотрит на пару (запрос, статья) вместе через self-attention, поэтому точнее bi-encoder.

**Почему не внедрил:** cross-encoder считает **каждую пару отдельно**, это дорого. `BAAI/bge-reranker-v2-m3` на 2 CPU непрактичен — не проходит даже по времени (таймаут на десятках пар), в проде это неприемлемая латентность на запрос. Осознанный инженерный размен: вместо тяжёлого второго этапа усилил bi-encoder (e5 → BGE-M3) + чанкинг и получил целевые 0.458 одноэтапным дешёвым пайплайном.

**Когда бы внедрил:** при наличии GPU — cross-encoder на топ-30 кандидатов (recall кандидатов высокий, значит есть куда расти именно за счёт порядка).

### Прочее (в порядке ожидаемой пользы)
- **Fine-tuning bi-encoder** на парах (запрос, статья) из `calibration.f` — контрастное обучение под домен Авито. Не делал: 500 запросов маловато и нужен аккуратный трейн/вал-сплит, чтобы не переобучиться.
- **Исправление опечаток** в запросах для лексической части — заметная доля запросов с опечатками («возрат», «прийдут»). Не делал: BGE-M3 и так устойчив к опечаткам, прирост неочевиден.
- **Query expansion** (расширение запроса синонимами) — потенциально помогает при разрыве словаря, но добавляет шум; отложил как менее приоритетное.